# 01 · Contrast prompt sets
Harmful prompts from Aya Red-teaming, harmless ones from the Aya Dataset, in English, French and Hindi.
Logic in `src/forged/data.py`; settings in `configs/contrast_sets.yaml`. CPU runtime is enough.

**Already done for team-doom** (repos `forged-fingerprints-contrast` and `forged-fingerprints-probes-secret`).
Rerunning with the same seed rebuilds the same splits; only rerun if you change the data.

In [ ]:
# ---- Setup: run first, in every new Colab session ----
# Colab secrets needed (key icon, Notebook access ON): HF_TOKEN (Hugging Face, write to team-doom)
# and GH_TOKEN (GitHub, read access to mathcrypto/forged-fingerprints while the repo is private).
import os, sys, subprocess
from google.colab import userdata

HF_ORG = 'team-doom'
REPO = 'mathcrypto/forged-fingerprints'
REPO_DIR = '/content/forged-fingerprints'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://{userdata.get("GH_TOKEN")}@github.com/{REPO}.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lexsi]'], check=True)

# Make the package importable now (no restart needed) and visible to !python scripts
sys.path.insert(0, f'{REPO_DIR}/src')
os.environ['PYTHONPATH'] = f'{REPO_DIR}/src'

from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
import forged
print('forged', forged.__version__, '| HF user:', whoami()['name'], '| cwd:', os.getcwd())

## 1. Check the Aya Dataset language names

In [ ]:
from forged.data import ContrastConfig, aya_language_counts
cfg = ContrastConfig.from_yaml('configs/contrast_sets.yaml')
counts = aya_language_counts()
for lang, lc in cfg.languages.items():
    print(lang, {n: counts.get(n, 0) for n in lc['aya_names']})

## 2. Build: load, screen, deduplicate, length-match, token check, split, hash

In [ ]:
from forged.data import build_contrast_sets
from forged.fingerprint import chat_token_counter
from forged.models import load_tokenizer
manifest = build_contrast_sets(cfg, token_counter=chat_token_counter(load_tokenizer('CohereLabs/tiny-aya-global')))
print(manifest['sizes'])
print('PROBE COMMITMENT (publish this):', manifest['probe_commitment_sha256'])

## 3. Read a sample (remove borderline prompts before pushing)

In [ ]:
import random
from forged.data import read_jsonl
rows = read_jsonl(cfg.out_dir / 'public' / 'A.jsonl')
for lang in cfg.languages:
    for label in ['harmful', 'harmless']:
        sub = [r['prompt'] for r in rows if r['language'] == lang and r['label'] == label]
        print(f'===== {lang} {label}')
        for p in random.Random(1).sample(sub, 5): print(' -', p[:160])

## 4. Push (two private repos)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
public_repo, secret_repo = f'{HF_ORG}/forged-fingerprints-contrast', f'{HF_ORG}/forged-fingerprints-probes-secret'
for repo in (public_repo, secret_repo):
    api.create_repo(repo, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path=str(cfg.out_dir / 'public'), repo_id=public_repo, repo_type='dataset',
                  commit_message=f'Contrast sets, seed {cfg.seed}')
api.upload_folder(folder_path=str(cfg.out_dir / 'secret'), repo_id=secret_repo, repo_type='dataset',
                  commit_message=f"Secret probes, commitment {manifest['probe_commitment_sha256'][:16]}")
print('Pushed (private):', public_repo, 'and', secret_repo)